In [1]:
import requests
import time
import os
from pathlib import Path

In [2]:
def recherche_zenodo(query, size=20): #size limite le nombre de requête
    url = "https://zenodo.org/api/records" # interrogation de zenodo
    filtered_hits = [] # récupération des hits
    current_page = 1 # on commence à la page 1

    # Paramètres de recherche
    params = {
        "q": query,
        "size": size,
        "all_versions": "false", # Filtre pour éviter les doublons de versions
        "type": "software", # Filtrage par type ("publication", "dataset", "software")
        "status": "published", # Impossible d'avoir des bons reviewed ?
        "sort": "mostrecent", # Filtrage par le plus récent en premier ("mostviewed", "-publicationdate")
    }

    # Boucle pour gérer la pagination
    while len(filtered_hits) < size:
        params["page"] = current_page
        response = requests.get(url, params=params) # lancement de la requête

        if response.status_code == 200: # code 200 = succès
          data = response.json()
          raw_hits = data["hits"]["hits"]

          if not raw_hits: # si la page est vide, on arrête
              break

          # Filtrage par extension
          new_hits = [
              h for h in raw_hits
              if any(f["key"].endswith((".xml", ".sbml", ".omex")) for f in h.get("files", []))
          ]

          filtered_hits.extend(new_hits)
          current_page += 1 # On passe à la page suivante pour le prochain tour
          time.sleep(1) # on met une pause entre les pages sinon Zenodo donne un code d'erreur
        else :
          print(f"Erreur API: {response.status_code}")

    # On ne retourne que le nombre demandé au départ
    final_results = filtered_hits[:size]
    print(f"{len(final_results)} modèles XML/SBML/OMEX valides récupérés pour {query}")
    return final_results

In [5]:
# Test pour une maladie du catalogue (ex: West Nile)
hits = recherche_zenodo("Dengue", 20)
for results in hits:
    title = results["metadata"]["title"]
    doi = results["links"]["doi"]
    print(f"Trouvé: {title} (DOI: {doi})")

0 modèles XML/SBML/OMEX valides récupérés pour Dengue


In [4]:
def file_extraction(doid) :
  url=f"https://zenodo.org/api/records/{doid}" # connexion à l'url
  response = requests.get(url) # requête
  if response.status_code == 200: # code 200 = succès
      data = response.json().get("files", []) # récupération des fichiers
      for f in data:
          download_url = f["links"]["self"]
          filename = f["key"]

          # Téléchargement du contenu
          r = requests.get(download_url)
          with open(filename, "wb") as local_file:
              local_file.write(r.content)
          print(f"Succès : {filename} enregistré.")

In [7]:
# Test de l'extraction d'un fichier
file_extraction(13960527)

Succès : SANN_PMML_Code_LCS12_Baseline_Final_Revised.xml enregistré.


In [11]:
def extraction_finale(liste):
    racine = Path("SBML_Zenodo") # on crée le premier dossier
    racine.mkdir(parents=True, exist_ok=True)
    os.chdir(racine) # on se place dans le dossier

    for virus in liste: # on teste chaque virus
        nom_dossier = virus[0] # création d'un dossier par virus
        dossier_virus = Path(nom_dossier)
        dossier_virus.mkdir(parents=True, exist_ok=True)
        os.chdir(dossier_virus) # on se place dans le dossier

        for autre_nom in virus: # teste de plusieurs variations de noms pour un même virus
            hits = recherche_zenodo(autre_nom, 20)
            for results in hits:
                doi = results["links"]["doi"].split(".")[-1] # extraction du DOI quand on a un hit
                file_extraction(doi) # récupération des fichiers à partir du DOI
        os.chdir("..") # on ressors du dossier

        if dossier_virus.exists() and not any(dossier_virus.iterdir()): # si dossier vide on supprime
            dossier_virus.rmdir()

    os.chdir("..") # retour hors du dossier Zenodo

In [12]:
extraction_finale([["dengue", "DENV"],["lyme","borrelia","borreliosis"],["mpox","monkeypox"],["west nile","WNV"],["influenza", "H5N1"],["tuberculosis", "TB OR mycobacterium"],["HIV"],["covid", "SARS-CoV-2"]])

0 modèles XML/SBML/OMEX valides récupérés pour dengue
0 modèles XML/SBML/OMEX valides récupérés pour DENV
0 modèles XML/SBML/OMEX valides récupérés pour lyme
0 modèles XML/SBML/OMEX valides récupérés pour borrelia
0 modèles XML/SBML/OMEX valides récupérés pour borreliosis
0 modèles XML/SBML/OMEX valides récupérés pour mpox
0 modèles XML/SBML/OMEX valides récupérés pour monkeypox
0 modèles XML/SBML/OMEX valides récupérés pour west nile
0 modèles XML/SBML/OMEX valides récupérés pour WNV
0 modèles XML/SBML/OMEX valides récupérés pour influenza
0 modèles XML/SBML/OMEX valides récupérés pour H5N1
0 modèles XML/SBML/OMEX valides récupérés pour tuberculosis
0 modèles XML/SBML/OMEX valides récupérés pour TB OR mycobacterium
0 modèles XML/SBML/OMEX valides récupérés pour HIV
6 modèles XML/SBML/OMEX valides récupérés pour covid
Succès : SANN_PMML_Code_LCS12_Baseline_Final_Revised.xml enregistré.
Succès : SANN_PMML_Code_Post COVID Syndrome at 3 Months.xml enregistré.
Succès : Final_SANN_PMML_Code